## EJECUCION

## NUM_EV

In [ ]:
import pandas as pd
import os
import sys
from tabulate import tabulate

def leer_archivo(ruta_archivo: str):
    # Verificar que el archivo existe
    if not os.path.exists(ruta_archivo):
        sys.exit(f"❌ El archivo no existe: {ruta_archivo}")

    # Detectar la extensión
    extension = os.path.splitext(ruta_archivo)[1].lower()

    try:
        if extension == ".csv":
            df = pd.read_csv(
                ruta_archivo,
                thousands=",",         # ✅ interpreta comas como miles (2022, 20000)
                na_values=["", " "],   # ✅ vacíos como NaN
                keep_default_na=True
            )
        elif extension in [".xls", ".xlsx"]:
            df = pd.read_excel(ruta_archivo)
        elif extension == ".json":
            df = pd.read_json(ruta_archivo)
        else:
            sys.exit(f"⚠️ Tipo de archivo no soportado: {extension}")

        # Llamada a la función para mostrar los primeros 10 datos
        mostrar_datos(df, 10)
        return df

    except Exception as e:
        sys.exit(f"❌ Error al leer el archivo: {e}")

def mostrar_datos(df, num_filas=10):
    """
    Muestra las primeras filas del DataFrame de una manera bonita.
    Por defecto, muestra las primeras 10 filas, pero se puede especificar un número diferente.
    
    :param df: El DataFrame a mostrar
    :param num_filas: El número de filas a mostrar (por defecto 10)
    """
    print(f"\n🔍 Mostrando las primeras {num_filas} filas del DataFrame:\n")
    
    # Mostrar las primeras 'num_filas' filas de manera bonita con tabulate
    print(tabulate(df.head(num_filas), headers='keys', tablefmt='pretty', showindex=False))

In [ ]:
import matplotlib.pyplot as plt

# Ruta del archivo
ruta = "./data/num_ev.csv"
df = leer_archivo(ruta)
df.head(10)

In [ ]:
# ---------------------------
# Limpieza de columnas numéricas
# ---------------------------
# Asegurar que algunas columnas sean numéricas
cols_numericas = ["AÑO_REGISTRO", "CAPACIDAD_PASAJEROS", "POTENCIA"]
for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.drop(columns=["ESTADO", "MODELO", "FECHA_REGISTRO", "CLASIFICACION", "LINEA", "CARROCERIA", "CILINDRAJE", "MODALIDAD", "ORGANISMO_TRANSITO", "CAPACIDAD_CARGA", "PESO", "EJES"])

df.to_csv('./data_limpieza/num_ev_limp.csv', index=False, encoding='utf-8')

df.head()

In [ ]:
import plotly.express as px

# Filtrar los datos para clasificar por los dos grupos de combustible
if df is not None and not df.empty:
    # Filtramos los datos para obtener solo vehículos eléctricos y los de otro combustible
    df_electricos = df[df['COMBUSTIBLE'] == 'ELECTRICO']
    df_otros = df[df['COMBUSTIBLE'] != 'ELECTRICO']

    # Contar vehículos eléctricos por año
    conteo_electricos = df_electricos['AÑO_REGISTRO'].value_counts().reset_index()
    conteo_electricos.columns = ['Año', 'Cantidad']
    conteo_electricos = conteo_electricos.sort_values('Año')

    # Contar vehículos de otros combustibles por año
    conteo_otros = df_otros['AÑO_REGISTRO'].value_counts().reset_index()
    conteo_otros.columns = ['Año', 'Cantidad']
    conteo_otros = conteo_otros.sort_values('Año')

    # Unir ambos DataFrames para tener una única visualización
    conteo_completo = pd.merge(conteo_electricos, conteo_otros, on='Año', how='outer', suffixes=('_Electrico', '_Hibridos'))
    conteo_completo.fillna(0, inplace=True)  # Rellenar valores nulos con 0

    # Crear gráfico dinámico con dos líneas, una para eléctricos y otra para Hibridos
    fig_completo = px.line(
        conteo_completo,
        x='Año',
        y=['Cantidad_Electrico', 'Cantidad_Hibridos'],
        labels={'Año': 'Año de registro', 'value': 'Cantidad de vehículos', 'variable': 'Tipo de Combustible'},
        title='📈 Evolución de Vehículos por Tipo de Combustible (Eléctricos vs Hibridos)'
    )

    # Personalización de estilo
    fig_completo.update_traces(
        mode='lines+markers',  # Mostrar líneas con puntos
        line=dict(width=3),
        marker=dict(size=9, line=dict(color='black', width=1)),
        textposition='top center'
    )

    # Ajustes visuales
    fig_completo.update_layout(
        height=500,
        xaxis_tickangle=-45,
        plot_bgcolor="white",
        xaxis=dict(showgrid=True, gridcolor="lightgray"),
        yaxis=dict(showgrid=True, gridcolor="lightgray"),
    )

    fig_completo.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de vehículos por tipo de combustible")



In [ ]:
# 🚗 Gráfica dinámica: Top 10 marcas de vehículos eléctricos
import plotly.express as px

if df is not None and not df.empty:
    # Obtener las 10 marcas más comunes
    top_marcas = (
        df["MARCA"]
        .value_counts()
        .head(10)
        .reset_index()
    )
    top_marcas.columns = ["Marca", "Cantidad"]

    print("🚗 Creando gráfica de Top 10 marcas de vehículos eléctricos...")

    # Crear gráfico de barras horizontal
    fig_marcas = px.bar(
        top_marcas,
        x="Cantidad",
        y="Marca",
        orientation="h",
        text="Cantidad",  # ✅ Etiquetas con valores
        title="🚗 Top 10 Marcas de Vehículos Eléctricos",
        labels={"Cantidad": "Cantidad de vehículos", "Marca": "Marca"},
        color="Marca",  # ✅ Colores diferenciados
        color_discrete_sequence=px.colors.qualitative.Vivid
    )

    # Personalización del estilo
    fig_marcas.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ Borde negro como las anteriores
    )

    # Ajustes de diseño
    fig_marcas.update_layout(
        height=500,
        xaxis_title="Cantidad de vehículos",
        yaxis_title="Marca",
        yaxis=dict(categoryorder="total ascending"),  # ✅ Orden ascendente (como en seaborn)
        plot_bgcolor="white",
        showlegend=False
    )

    fig_marcas.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de marcas")


In [ ]:
# 🛻 Gráfica dinámica: Distribución por clase de vehículo
import plotly.express as px

if df is not None and not df.empty:
    # ✅ Calcular distribución de clases
    class_counts = df["CLASE"].value_counts()

    # ✅ Agrupar clases menores al 1% en "Otros"
    threshold = class_counts.sum() * 0.01
    small_classes = class_counts[class_counts < threshold].index
    df["CLASE"] = df["CLASE"].replace(small_classes, "Otros")

    # ✅ Recalcular distribución
    class_counts = df["CLASE"].value_counts().reset_index()
    class_counts.columns = ["Clase", "Cantidad"]

    print("🛻 Creando gráfica de distribución por clase de vehículo...")

    # ✅ Crear gráfico circular con Plotly
    fig_clase = px.pie(
        class_counts,
        names="Clase",
        values="Cantidad",
        title="🛻 Distribución por Clase de Vehículo",
        color="Clase",
        color_discrete_sequence=px.colors.qualitative.Set2,  # 🎨 Colores suaves y agradables
        hole=0.3  # 🔘 estilo donut (más moderno)
    )

    # ✅ Ajustes visuales
    fig_clase.update_traces(
        textposition="inside",
        textinfo="percent+label",  # muestra porcentaje y nombre
        marker=dict(line=dict(color="black", width=1))  # borde negro estilo uniforme
    )

    fig_clase.update_layout(
        height=500,
        showlegend=True,
        legend_title_text="Clase de Vehículo",
        plot_bgcolor="white"
    )

    fig_clase.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de clases de vehículo")


In [ ]:
# ⚙️ Gráfica dinámica: Potencia promedio por marca (Top 10)
import plotly.express as px

if df is not None and not df.empty and "POTENCIA" in df.columns:
    # ✅ Calcular promedio de potencia por marca
    potencia_marca = (
        df.groupby("MARCA")["POTENCIA"]
        .mean()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    print("⚙️ Creando gráfica de potencia promedio por marca...")

    # ✅ Crear gráfica de barras dinámica
    fig_potencia = px.bar(
        potencia_marca,
        x="MARCA",
        y="POTENCIA",
        title="⚙️ Potencia Promedio por Marca (Top 10)",
        labels={"MARCA": "Marca", "POTENCIA": "Potencia Promedio (HP o kW)"},
        text="POTENCIA",
        color="MARCA",
        color_discrete_sequence=px.colors.qualitative.Pastel1
    )

    # ✅ Personalización visual
    fig_potencia.update_traces(
        texttemplate="%{text:.2f}",  # muestra potencia con dos decimales
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_potencia.update_layout(
        height=500,
        xaxis_tickangle=-45,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_potencia.show()

else:
    print("❌ No hay datos disponibles o la columna 'POTENCIA' no existe para crear la gráfica")


In [ ]:
# 🧍‍♂️ Gráfica dinámica: Capacidad promedio de pasajeros por tipo de servicio
import plotly.express as px

if (
    df is not None 
    and not df.empty 
    and "CAPACIDAD_PASAJEROS" in df.columns 
    and "SERVICIO" in df.columns
):
    # ✅ Calcular capacidad promedio de pasajeros por servicio
    capacidad_servicio = (
        df.groupby("SERVICIO")["CAPACIDAD_PASAJEROS"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    print("🧍‍♂️ Creando gráfica de capacidad promedio de pasajeros por servicio...")

    # ✅ Crear gráfica de barras dinámica
    fig_capacidad = px.bar(
        capacidad_servicio,
        x="SERVICIO",
        y="CAPACIDAD_PASAJEROS",
        title="🧍‍♂️ Capacidad Promedio de Pasajeros por Tipo de Servicio",
        labels={
            "SERVICIO": "Tipo de Servicio",
            "CAPACIDAD_PASAJEROS": "Promedio de Pasajeros",
        },
        text="CAPACIDAD_PASAJEROS",
        color="SERVICIO",
        color_discrete_sequence=px.colors.qualitative.Set3  # 🎨 paleta consistente con tus otras gráficas
    )

    # ✅ Personalización visual
    fig_capacidad.update_traces(
        texttemplate="%{text:.1f}",
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_capacidad.update_layout(
        height=500,
        xaxis_tickangle=-30,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_capacidad.show()

else:
    print("❌ No hay datos disponibles o faltan columnas necesarias ('CAPACIDAD_PASAJEROS', 'SERVICIO') para crear la gráfica.")


In [ ]:
# 🌎 Gráfica dinámica: Top 3 departamentos con más vehículos
import plotly.express as px

if df is not None and not df.empty and "DEPARTAMENTO" in df.columns:
    # ✅ Contar vehículos por departamento
    departamento_counts = df["DEPARTAMENTO"].value_counts().reset_index()
    departamento_counts.columns = ["Departamento", "Cantidad"]

    # ✅ Separar el Top 3
    top3 = departamento_counts.head(3)

    print("🌎 Creando gráfica del Top 3 departamentos con más vehículos...")

    # ✅ Crear gráfica de barras dinámica
    fig_top3 = px.bar(
        top3,
        x="Departamento",
        y="Cantidad",
        title="🌎 Top 3 Departamentos con Más Vehículos",
        labels={"Departamento": "Departamento", "Cantidad": "Cantidad de Vehículos"},
        text="Cantidad",
        color="Departamento",
        color_discrete_sequence=px.colors.qualitative.G10
    )

    # ✅ Personalización visual
    fig_top3.update_traces(
        texttemplate="%{text}",
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_top3.update_layout(
        height=500,
        yaxis_range=[0, top3["Cantidad"].max() * 1.2],  # Espacio extra arriba
        xaxis_tickangle=-15,
        showlegend=False,
        plot_bgcolor="white"
    )

    fig_top3.show()

else:
    print("❌ No hay datos disponibles o falta la columna 'DEPARTAMENTO' para crear la gráfica.")


In [ ]:
# 🚗 Gráfica dinámica: Resto de departamentos con vehículos
import plotly.express as px

if df is not None and not df.empty and "DEPARTAMENTO" in df.columns:
    # ✅ Contar vehículos por departamento
    departamento_counts = df["DEPARTAMENTO"].value_counts().reset_index()
    departamento_counts.columns = ["Departamento", "Cantidad"]

    # ✅ Separar el resto (excluyendo el Top 3)
    resto = departamento_counts.iloc[3:]

    print("🚗 Creando gráfica del resto de departamentos...")

    # ✅ Crear gráfica de barras dinámica
    fig_resto = px.bar(
        resto,
        x="Departamento",
        y="Cantidad",
        title="🚗 Resto de Departamentos",
        labels={"Departamento": "Departamento", "Cantidad": "Cantidad de Vehículos"},
        text="Cantidad",
        color="Departamento",
        color_discrete_sequence=px.colors.qualitative.G10  # Paleta coherente
    )

    # ✅ Personalización visual
    fig_resto.update_traces(
        texttemplate="%{text}",
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_resto.update_layout(
        height=600,
        yaxis_range=[0, resto["Cantidad"].max() * 1.2],
        xaxis_tickangle=-90,
        showlegend=False,
        plot_bgcolor="white",
        margin=dict(l=40, r=40, t=80, b=150)
    )

    fig_resto.show()

else:
    print("❌ No hay datos disponibles o falta la columna 'DEPARTAMENTO' para crear la gráfica.")

In [ ]:
if df is not None and not df.empty:
    # Contar la cantidad de vehículos por tipo de servicio
    servicios = (
        df["SERVICIO"]
        .value_counts()
        .reset_index()
    )
    servicios.columns = ["Servicio", "Cantidad"]

    print("⚡ Creando gráfica de cantidad de vehículos eléctricos por tipo de servicio...")

    # Crear gráfico de barras horizontal
    fig_servicio = px.bar(
        servicios,
        x="Cantidad",
        y="Servicio",
        orientation="h",
        text="Cantidad",  # ✅ Mostrar los valores sobre las barras
        title="⚡ Cantidad de Vehículos Eléctricos por Tipo de Servicio",
        labels={"Cantidad": "Cantidad de vehículos", "Servicio": "Tipo de servicio"},
        color="Servicio",  # ✅ Colores distintos para cada categoría
        color_discrete_sequence=px.colors.qualitative.Vivid
    )

    # Personalización de trazos
    fig_servicio.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ Borde negro
    )

    # Ajustes de diseño y estilo
    fig_servicio.update_layout(
        height=500,
        xaxis_title="Cantidad de vehículos",
        yaxis_title="Tipo de servicio",
        yaxis=dict(categoryorder="total ascending"),  # ✅ Orden ascendente
        plot_bgcolor="white",
        showlegend=False,
        title_font=dict(size=22),
        font=dict(size=14)
    )

    fig_servicio.show()

## USO_ESTACIONES

In [ ]:
# Ruta del archivo
ruta = "./data/uso_estaciones.csv"
df = leer_archivo(ruta)

In [ ]:
# Convertir columnas de consumo a numéricas, manejando errores
df["TOTAL DE CONSUMO"] = pd.to_numeric(df["TOTAL DE CONSUMO"], errors="coerce")
# Eliminar columnas 'Consumo Primera Mitad' y 'Consumo Segunda Mitad' que ya no son necesarias
df = df.drop(columns=["CONSUMO PRIMERA MITAD DEL MES", "CONSUMO SEGUNDA MITAD DEL MES"])
df.to_csv('./data_limpieza/uso_estaciones_limp.csv', index=False, encoding='utf-8')
mostrar_datos(df, 10)

In [ ]:
# ---------------------------
# 6. Consumo total mensual (versión Plotly Express)
# ---------------------------
import plotly.express as px
import pandas as pd

if "MES" in df.columns and "AÑO" in df.columns and "TOTAL DE CONSUMO" in df.columns:
    # Crear copia del DataFrame original
    drf = df.copy()

    # Normalizar columnas
    drf["AÑO_norm"] = drf["AÑO"].astype(str).str.replace(r"\D", "", regex=True)
    drf["MES_norm"] = drf["MES"].astype(str).str.strip().str.lower()

    # Filtrar: eliminar filas de junio a octubre de 2020
    meses_a_eliminar = ["junio", "julio", "agosto", "septiembre", "octubre"]
    mask_eliminar = (drf["AÑO_norm"] == "2020") & (drf["MES_norm"].isin(meses_a_eliminar))
    drf = drf[~mask_eliminar].copy()

    # Convertir el consumo total a número
    drf["TOTAL DE CONSUMO"] = (
        drf["TOTAL DE CONSUMO"].astype(str).str.replace(",", "", regex=True).str.strip()
    ).astype(float)

    # Crear columna combinada de periodo (Mes + Año)
    drf["Periodo"] = drf["MES"].astype(str).str.strip() + " " + drf["AÑO_norm"]
    drf["Periodo"] = pd.Categorical(drf["Periodo"], categories=drf["Periodo"].unique(), ordered=True)

    # Crear gráfica
    fig_consumo = px.line(
        drf,
        x="Periodo",
        y="TOTAL DE CONSUMO",
        title="📈 Consumo Total Mensual",
        labels={"Periodo": "Mes y Año", "TOTAL DE CONSUMO": "Consumo Total"},
        markers=True,
        line_shape="linear",
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    # Personalización visual
    fig_consumo.update_traces(
        text=drf["TOTAL DE CONSUMO"].apply(lambda x: f"{int(x):,}" if pd.notnull(x) else ""),
        textposition="top center",
        hovertemplate="<b>%{x}</b><br>Consumo: %{y:,}"
    )

    fig_consumo.update_layout(
        height=500,
        xaxis_tickangle=-45,
        font=dict(size=12),
        plot_bgcolor="white",
        hovermode="x unified",
        margin=dict(l=40, r=40, t=80, b=80)
    )

    fig_consumo.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray")
    fig_consumo.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", tickformat=",d")

    fig_consumo.show()

else:
    print("❌ No se encontraron las columnas necesarias ('MES', 'AÑO', 'TOTAL DE CONSUMO').")


In [ ]:
# ---------------------------
# 7. Consumo Promedio Anual (versión Plotly Express)
# ---------------------------
import plotly.express as px

if "AÑO" in df.columns and "TOTAL DE CONSUMO" in df.columns:
    # Asegurar tipo string para el eje X
    df["AÑO"] = df["AÑO"].astype(str)

    # Calcular consumo promedio anual
    consumo_anual = df.groupby("AÑO", as_index=False)["TOTAL DE CONSUMO"].mean()

    print("📊 Creando gráfica de consumo promedio anual...")

    fig_consumo_anual = px.bar(
        consumo_anual,
        x="AÑO",
        y="TOTAL DE CONSUMO",
        title="📊 Consumo Promedio Anual",
        labels={
            "AÑO": "Año",
            "TOTAL DE CONSUMO": "Consumo Promedio"
        },
        text="TOTAL DE CONSUMO",
        color="AÑO",
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    # Personalizar etiquetas y formato
    fig_consumo_anual.update_traces(
        texttemplate="%{y:,.0f}",
        textposition="outside",
        marker_line_color="black",
        marker_line_width=1
    )

    # Ajustes visuales del layout
    fig_consumo_anual.update_layout(
        height=500,
        xaxis_tickangle=-30,
        font=dict(size=12),
        plot_bgcolor="white",
        showlegend=False,
        margin=dict(l=40, r=40, t=80, b=60)
    )

    # Líneas de cuadrícula suaves
    fig_consumo_anual.update_xaxes(showgrid=False)
    fig_consumo_anual.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", tickformat=",d")

    fig_consumo_anual.show()

else:
    print("❌ No se encontraron las columnas necesarias ('AÑO', 'TOTAL DE CONSUMO').")


## ESTACIONES

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-muted")  # Estilo limpio


# Ruta del archivo
ruta = "./data/estaciones.csv"
df = leer_archivo(ruta)

In [ ]:
# Eliminar columnas 'Consumo Primera Mitad' y 'Consumo Segunda Mitad' que ya no son necesarias
df = df.drop(columns=["Tipo de estacion", "Ubicación o sitio web", "Latitud", "Longitud"])
df.to_csv('./data_limpieza/estaciones_limp.csv', index=False, encoding='utf-8')
mostrar_datos(df, 10)

In [ ]:
# 📊 Gráfica dinámica: Cantidad de estaciones por ciudad

if df is not None and not df.empty:
    conteo_ciudad = df["Ciudad"].value_counts().reset_index()
    conteo_ciudad.columns = ["Ciudad", "Cantidad"]

    print("📊 Creando gráfica de estaciones por ciudad...")

    fig_ciudades = px.bar(
        conteo_ciudad,
        x="Ciudad",
        y="Cantidad",
        title="📊 Cantidad de Estaciones por Ciudad",
        labels={"Ciudad": "Ciudad", "Cantidad": "Número de Estaciones"},
        text="Cantidad",  # ✅ Muestra las etiquetas encima de las barras
        color="Ciudad",   # ✅ Colores diferenciados por ciudad
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    fig_ciudades.update_traces(
        textposition="outside", 
        marker=dict(line=dict(color="black", width=1))  # ✅ Borde negro estilo matplotlib
    )

    fig_ciudades.update_layout(
        height=500,
        xaxis_tickangle=-45,  # ✅ Rotar etiquetas del eje X como en matplotlib
        showlegend=False      # ✅ Ocultar leyenda innecesaria
    )

    fig_ciudades.show()
else:
    print("❌ No hay datos disponibles para crear la gráfica de ciudades")


In [ ]:
# 📊 Gráfica dinámica: Distribución de tipo de carga (agrupando "Información no disponible" con "Semi Rápida")
if df is not None and not df.empty:
    # Reemplazar valores de tipo de carga
    df["Tipo de carga"] = df["Tipo de carga"].replace({
        "Información no disponible": "Semi Rápida"
    })

    # Agrupar nuevamente con los valores actualizados
    conteo_tipo_carga = df["Tipo de carga"].value_counts().reset_index()
    conteo_tipo_carga.columns = ["Tipo de carga", "Cantidad"]

    print("🔌 Creando gráfica de distribución por tipo de carga (agrupada)...")

    fig_carga = px.bar(
        conteo_tipo_carga,
        x="Tipo de carga",
        y="Cantidad",
        title="🔌 Distribución de Tipo de Carga",
        labels={"Tipo de carga": "Tipo de Carga", "Cantidad": "Número de Estaciones"},
        text="Cantidad",
        color="Tipo de carga",
        color_discrete_sequence=["mediumseagreen", "lightgreen", "darkgreen"]
    )

    fig_carga.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_carga.update_layout(
        height=450,
        xaxis_tickangle=-30,
        showlegend=False
    )

    fig_carga.show()

else:
    print("❌ No hay datos disponibles para crear la gráfica de tipo de carga")


In [ ]:
# 📊 Gráfica dinámica: Frecuencia de estándares de cargador
if df is not None and not df.empty:
    conteo_cargadores = df["Estándar Cargador"].value_counts().reset_index()
    conteo_cargadores.columns = ["Estándar Cargador", "Cantidad"]

    print("⚡ Creando gráfica de estándares de cargador...")

    fig_cargadores = px.bar(
        conteo_cargadores,
        x="Estándar Cargador",
        y="Cantidad",
        title="⚡ Frecuencia de Estándares de Cargador",
        labels={"Estándar Cargador": "Estándar", "Cantidad": "Cantidad"},
        text="Cantidad",  # ✅ etiquetas encima de cada barra
        color="Estándar Cargador",
        color_discrete_sequence=px.colors.qualitative.Prism  # ✅ paleta más variada tipo 'orchid'
    )

    fig_cargadores.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ borde negro estilo matplotlib
    )

    fig_cargadores.update_layout(
        height=500,
        xaxis_tickangle=-45,
        xaxis=dict(tickfont=dict(size=10)),  # ✅ etiquetas más legibles si son largas
        showlegend=False
    )

    fig_cargadores.show()
else:
    print("❌ No hay datos disponibles para crear la gráfica de estándares de cargador")


In [ ]:
# 📊 Gráfica dinámica: Horarios de funcionamiento de estaciones
if df is not None and not df.empty:
    conteo_horarios = df["Horario"].value_counts().reset_index()
    conteo_horarios.columns = ["Horario", "Cantidad"]

    print("🕒 Creando gráfica de horarios de funcionamiento...")

    fig_horarios = px.bar(
        conteo_horarios,
        x="Horario",
        y="Cantidad",
        title="🕒 Horarios de Funcionamiento de Estaciones",
        labels={"Horario": "Horario", "Cantidad": "Cantidad"},
        text="Cantidad",  # ✅ etiquetas encima de cada barra
        color="Horario",
        color_discrete_sequence=px.colors.qualitative.Set2  # ✅ paleta cálida tipo coral
    )

    fig_horarios.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))  # ✅ borde negro estilo matplotlib
    )

    fig_horarios.update_layout(
        height=500,
        xaxis_tickangle=-45,
        xaxis=dict(tickfont=dict(size=10)),  # ✅ mejor legibilidad
        showlegend=False
    )

    fig_horarios.show()
else:
    print("❌ No hay datos disponibles para crear la gráfica de horarios")


In [ ]:
# 📍 Mapa dinámico de estaciones de carga eléctrica en Antioquia
import plotly.express as px

if df is not None and not df.empty:
    # Separar coordenadas si no existen como columnas numéricas
    if "Lat" not in df.columns or "Lon" not in df.columns:
        df[["Lat", "Lon"]] = df["Coordenadas"].str.split(",", expand=True)
        df["Lat"] = df["Lat"].astype(str).str.replace(",", ".").astype(float)
        df["Lon"] = df["Lon"].astype(str).str.replace(",", ".").astype(float)

    # Filtrar área de Antioquia (igual que antes)
    df_valid = df[
        (df["Lat"] >= 5.5) & (df["Lat"] <= 7.5) &
        (df["Lon"] >= -76.2) & (df["Lon"] <= -75.0)
    ].copy()

    if not df_valid.empty:
        print("🗺️ Creando mapa dinámico de estaciones de carga en Antioquia...")

        fig_map = px.scatter_mapbox(
            df_valid,
            lat="Lat",
            lon="Lon",
            hover_name="Estación",     # ✅ nombre de la estación en tooltip
            hover_data={"Lat": True, "Lon": True, "Ciudad": True, "Horario": True},
            color_discrete_sequence=["dodgerblue"],
            zoom=10.5,
            height=800,

        )
        # Aumentar el tamaño de los puntos
        fig_map.update_traces(marker=dict(size=15))  # 🔧 Cambia aquí el tamaño (ajusta a gusto)

        fig_map.update_layout(
            mapbox_style="open-street-map",  # ✅ estilo libre de Mapbox
            title="🗺️ Ubicación de Estaciones de Carga Eléctrica (Antioquia - Colombia)",
            margin={"r":0, "t":40, "l":0, "b":0}
        )

        fig_map.show()
    else:
        print("⚠️ No hay estaciones dentro del rango de Antioquia.")
else:
    print("❌ No hay datos disponibles para crear el mapa")


## Estaciones mundiales

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-muted")  # Estilo limpio


# Ruta del archivo
ruta = "./data/estaciones_mundiales.csv"
df = leer_archivo(ruta)

In [ ]:
import plotly.express as px
# Asegúrate de que df ya ha sido cargado en este punto, por ejemplo:
# df = leer_archivo("./data/estaciones_mundiales.csv")

# --- Variables de Columna ---
X_COL = "País"
Y_COL = "Total de Estaciones de Carga (públicas)"

if df is not None and not df.empty:
    
    # Ordenar el DataFrame para que la gráfica muestre las barras de mayor a menor
    df = df.sort_values(by=Y_COL, ascending=False)
    
    print("📊 Creando gráfica de Total de Estaciones de Carga por País...")

    fig_estaciones = px.bar(
        df,
        x=X_COL,
        y=Y_COL,
        title="🔌 Total de Estaciones de Carga Públicas por País",
        labels={X_COL: "País", Y_COL: "Total de Estaciones"},
        text=Y_COL,  # ✅ Muestra las etiquetas encima de cada barra
        color=X_COL,
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    fig_estaciones.update_traces(
        textposition="outside",
        marker=dict(line=dict(color="black", width=1))
    )

    fig_estaciones.update_layout(
        height=500,
        xaxis_tickangle=-45,
        xaxis=dict(tickfont=dict(size=10)),
        showlegend=False
    )

    # Muestra la gráfica en tu entorno interactivo (Jupyter/Colab)
    fig_estaciones.show()
    
    # Si lo necesitas guardar para usarlo en un dashboard o web:
    # fig_estaciones.write_html("grafico_estaciones_mundiales.html")
    print("✅ Gráfica generada exitosamente.")
else:
    print("❌ No hay datos disponibles en el DataFrame para crear la gráfica.")

## MEXICO 

In [ ]:
# ==========================================================
# 📦 CELDA 1: Importación de Librerías
# ==========================================================
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Configurar Plotly para que se muestren las gráficas inline
import plotly.io as pio
pio.renderers.default = "notebook"

print("✅ Librerías importadas correctamente")


In [ ]:
# ==========================================================
# 📂 CELDA 2: Configuración de Ruta y Carga de Datos
# ==========================================================
ruta = "../data/conjunto_de_datos"

def cargar_datos():
    """Función de carga definida en la siguiente celda"""
    pass

# Llamar función (se definirá más adelante)
df = cargar_datos()

## 📈 INDICADORES CLAVE DE RENDIMIENTO (KPIs)

En esta sección analizaremos los principales indicadores del mercado de vehículos alternativos:
- Total de vehículos eléctricos vendidos
- Total de vehículos híbridos
- Total de vehículos plug-in

In [ ]:
# ==========================================================
# 📥 CELDA 3: Carga y concatenación de archivos CSV
# ==========================================================
ruta = "./data/conjunto_de_datos"

def cargar_datos():
    """Carga todos los archivos CSV del directorio indicado y los concatena."""
    try:
        if not os.path.exists(ruta):
            print(f"❌ La carpeta {ruta} no existe.")
            return None

        archivos = [f for f in os.listdir(ruta) if f.endswith(".csv")]
        if not archivos:
            print("⚠️ No se encontraron archivos CSV en la carpeta.")
            return None

        print(f"📁 Archivos encontrados ({len(archivos)}): {archivos}")
        df_list = []

        for archivo in archivos:
            path = os.path.join(ruta, archivo)
            for encoding in ['utf-8', 'latin-1', 'iso-8859-1']:
                try:
                    df_temp = pd.read_csv(path, encoding=encoding)
                    df_list.append(df_temp)
                    print(f"✅ {archivo} cargado correctamente ({len(df_temp)} filas)")
                    break
                except UnicodeDecodeError:
                    continue
                except Exception as e:
                    print(f"⚠️ Error leyendo {archivo}: {e}")

        if not df_list:
            print("❌ No se pudieron cargar archivos válidos.")
            return None

        df_final = pd.concat(df_list, ignore_index=True)
        print(f"\n🎉 Datos combinados: {len(df_final)} filas, {len(df_final.columns)} columnas")
        return df_final

    except Exception as e:
        print(f"❌ Error inesperado: {e}")
        return None


# Cargar datos
df = cargar_datos()

# Mostrar vista rápida
if df is not None:
    print("\n📊 Vista previa de los datos:")
    display(df.head())
    print(f"\n📋 Columnas detectadas: {list(df.columns)}")

In [ ]:
# ==========================================================
# 🧹 CELDA 4: Limpieza y Procesamiento del Dataset
# ==========================================================
def procesar_datos(df):
    """Limpia y transforma el dataset para análisis posterior."""
    if df is None or df.empty:
        print("❌ No hay datos para procesar.")
        return None

    print("🔧 Procesando datos...")

    df = df.copy()

    # --- Limpieza general ---
    # Eliminar duplicados
    filas_antes = len(df)
    df.drop_duplicates(inplace=True)
    print(f"🧩 Duplicados eliminados: {filas_antes - len(df)}")

    # --- Conversiones ---
    for col in ["ANIO", "ID_MES"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            print(f"⚠️ Columna {col} no encontrada. Se creará con valores nulos.")
            df[col] = np.nan

    # Eliminar filas sin año o mes
    filas_antes = len(df)
    df.dropna(subset=["ANIO", "ID_MES"], inplace=True)
    print(f"📉 Filas eliminadas por valores nulos en ANIO/ID_MES: {filas_antes - len(df)}")

    # --- Creación de fecha ---
    try:
        df["FECHA"] = pd.to_datetime(df["ANIO"].astype(int).astype(str) +
                                     df["ID_MES"].astype(int).astype(str).str.zfill(2),
                                     format="%Y%m", errors='coerce')
        df["MES_NOMBRE"] = df["FECHA"].dt.strftime("%b")
        df["TRIMESTRE"] = df["FECHA"].dt.quarter
        print("✅ Columnas de fecha generadas correctamente")
    except Exception as e:
        print(f"⚠️ Error generando fechas: {e}")

    # --- Columnas de vehículos ---
    columnas_vehiculos = ["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"]
    for col in columnas_vehiculos:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            print(f"⚠️ Columna {col} no encontrada. Se llenará con ceros.")
            df[col] = 0

    # Crear total alternativo
    df["TOTAL_ALT"] = df[columnas_vehiculos].sum(axis=1)

    print("✅ Procesamiento completado correctamente")
    return df


# Procesar los datos cargados
if df is not None:
    df = procesar_datos(df)
    if df is not None:
        print("\n📊 Estadísticas generales:")
        display(df.describe())
        print("\n📋 Muestra de datos procesados:")
        display(df[["ANIO", "ID_MES", "VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN", "TOTAL_ALT"]].head(10))

        # Guardar el DataFrame limpio a CSV


        salida = "./data_limpieza/BD_Mexico_limp.csv"  # Ruta y nombre del archivo de salida
        df.to_csv(salida, index=False, encoding="utf-8")
        print(f"\n✅ Archivo CSV limpio guardado exitosamente en: {salida}")


## 📈 INDICADORES CLAVE (KPIs)

In [ ]:
# ==========================================================
# 📈 CELDA 5: Indicadores Clave (KPIs)
# ==========================================================
print("="*60)
print("📊 INDICADORES CLAVE")
print("="*60)

if df is not None and not df.empty:
    total_electricos = df["VEH_ELECTR"].sum()
    total_hibridos = df["VEH_HIBRIDAS"].sum()
    total_plugin = df["VEH_HIBRIDAS_PLUGIN"].sum()
    total_general = df["TOTAL_ALT"].sum()

    # Mostrar tabla de KPIs
    kpis = pd.DataFrame({
        'Indicador': ['Total Eléctricos', 'Total Híbridos', 'Total Plug-in', 'Total General'],
        'Valor': [f"{total_electricos:,}", f"{total_hibridos:,}", f"{total_plugin:,}", f"{total_general:,}"]
    })
    display(kpis)

    # Gráfica dinámica
    if total_general > 0:
        fig = px.bar(
            x=['Eléctricos', 'Híbridos', 'Plug-in'],
            y=[total_electricos, total_hibridos, total_plugin],
            title="🚗 Resumen General de Vehículos por Tipo",
            labels={'x': 'Tipo de Vehículo', 'y': 'Cantidad Total'},
            color=['Eléctricos', 'Híbridos', 'Plug-in'],
            color_discrete_map={
                'Eléctricos': '#2E8B57',
                'Híbridos': '#FF6347',
                'Plug-in': '#1E90FF'
            },
            text=[total_electricos, total_hibridos, total_plugin]
        )
        fig.update_traces(texttemplate='%{text:,}', textposition='outside')
        fig.update_layout(height=420, showlegend=False, yaxis_title="Cantidad", xaxis_title="Tipo")
        fig.show()
    else:
        print("⚠️ No hay datos suficientes para generar la gráfica.")
else:
    print("❌ No se pudieron calcular los KPIs. Verifique los datos cargados.")


In [ ]:
def procesar_datos(df):
    """
    Limpia y procesa el dataset para el análisis
    """
    if df is None:
        print("❌ No hay datos para procesar")
        return None
        
    print("🔧 Procesando datos...")
    
    # Hacer una copia para no modificar el original
    df = df.copy()
    
    # Conversiones de tipos
    df["ANIO"] = pd.to_numeric(df["ANIO"], errors='coerce')
    df["ID_MES"] = pd.to_numeric(df["ID_MES"], errors='coerce')
    
    # Eliminar filas con valores nulos en columnas críticas
    filas_antes = len(df)
    df = df.dropna(subset=["ANIO", "ID_MES"])
    print(f"📉 Filas eliminadas por valores nulos: {filas_antes - len(df)}")
    
    # Crear columna de fecha
    try:
        df["FECHA"] = pd.to_datetime(df["ANIO"].astype(int).astype(str) + 
                                     df["ID_MES"].astype(int).astype(str).str.zfill(2), 
                                     format="%Y%m", errors='coerce')
        
        df["MES_NOMBRE"] = df["FECHA"].dt.strftime("%b")
        df["TRIMESTRE"] = df["FECHA"].dt.quarter
        print("✅ Fechas procesadas correctamente")
    except Exception as e:
        print(f"⚠️ Error procesando fechas: {e}")
    
    # Asegurar que las columnas de vehículos sean numéricas
    columnas_vehiculos = ["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"]
    for col in columnas_vehiculos:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            print(f"✅ Columna {col} procesada")
        else:
            print(f"⚠️ Columna {col} no encontrada")
            df[col] = 0  # Crear columna con ceros si no existe
    
    # Crear total de vehículos alternativos
    df["TOTAL_ALT"] = df[columnas_vehiculos].sum(axis=1)
    
    print("✅ Procesamiento completado")
    return df

# Procesar los datos solo si se cargaron correctamente
if df is not None:
    df = procesar_datos(df)
    if df is not None:
        print(f"\n📊 Dataset procesado:")
        display(df.describe())
        print(f"\n📋 Muestra de datos procesados:")
        display(df[["ANIO", "ID_MES", "VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN", "TOTAL_ALT"]].head(10))
else:
    print("❌ No se pueden procesar los datos porque no se cargaron correctamente")

# ## 📊 ANÁLISIS DE TENDENCIAS

In [ ]:
if not df.empty:
    print("📈 Creando gráfica de evolución anual...")
    
    ventas_anuales = df.groupby("ANIO")[["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"]].sum().reset_index()
    
    fig1 = px.line(
        ventas_anuales, 
        x="ANIO", 
        y=["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"],
        title="📈 Evolución Anual de Ventas por Tipo de Vehículo",
        labels={"value": "Número de Vehículos", "variable": "Tipo de Vehículo", "ANIO": "Año"},
        color_discrete_map={
            "VEH_ELECTR": "#2E8B57",
            "VEH_HIBRIDAS_PLUGIN": "#1E90FF", 
            "VEH_HIBRIDAS": "#FF6347"
        }
    )
    fig1.update_traces(mode='lines+markers', line=dict(width=3))
    fig1.update_layout(
        height=500,
        legend=dict(
            title="Tipo de Vehículo",
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    fig1.show()
    
    # Mostrar tabla de datos
    print("\n📊 Datos de evolución anual:")
    display(ventas_anuales)

In [ ]:
# 📊 Gráfica 2: Análisis mensual/trimestral
if not df.empty:
    años_unicos = len(df["ANIO"].unique())
    
    if años_unicos <= 3:
        print("📅 Creando análisis mensual (pocos años)...")
        ventas_mensuales = df.groupby(["ANIO", "ID_MES", "MES_NOMBRE"])[["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"]].sum().reset_index()
        
        fig2 = px.line(
            ventas_mensuales, 
            x="ID_MES", 
            y="VEH_ELECTR", 
            color="ANIO",
            title="📅 Tendencia Mensual - Vehículos Eléctricos",
            labels={"VEH_ELECTR": "Vehículos Eléctricos", "ID_MES": "Mes"}
        )
        fig2.update_traces(mode='lines+markers')
    else:
        print("📅 Creando análisis trimestral (muchos años)...")
        ventas_trimestrales = df.groupby(["ANIO", "TRIMESTRE"])[["VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN"]].sum().reset_index()
        
        fig2 = px.bar(
            ventas_trimestrales,
            x="ANIO",
            y=["VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN"],
            title="📅 Distribución Trimestral de Ventas",
            barmode="group",
            labels={"value": "Número de Vehículos", "variable": "Tipo de Vehículo"}
        )
    
    fig2.update_layout(height=500)
    fig2.show()

# ## 🎯 ANÁLISIS COMPARATIVO

In [ ]:
# 🥧 Gráfica 3: Composición por tipo (pie chart)
if not df.empty:
    print("🥧 Creando gráfica de composición por tipo...")
    
    totales_tipo = {
        "Eléctricos": df["VEH_ELECTR"].sum(),
        "Híbridos": df["VEH_HIBRIDAS"].sum(),
        "Híbridos Plug-in": df["VEH_HIBRIDAS_PLUGIN"].sum()
    }
    
    fig3 = px.pie(
        values=list(totales_tipo.values()),
        names=list(totales_tipo.keys()),
        title="🥧 Distribución Porcentual por Tipo de Vehículo",
        color_discrete_map={
            "Eléctricos": "#2E8B57",
            "Híbridos Plug-in": "#1E90FF",
            "Híbridos": "#FF6347"
        }
    )
    fig3.update_traces(textposition='inside', textinfo='percent+label')
    fig3.update_layout(height=500)
    fig3.show()
    
    # Mostrar porcentajes exactos
    total = sum(totales_tipo.values())
    print("\n📊 Distribución porcentual exacta:")
    for tipo, valor in totales_tipo.items():
        porcentaje = (valor / total) * 100 if total > 0 else 0
        print(f"   {tipo}: {valor:,} ({porcentaje:.1f}%)")

In [ ]:
# 📈 Gráfica 4: Crecimiento año a año
if not df.empty and len(ventas_anuales) > 1:
    print("📈 Creando análisis de crecimiento...")
    
    ventas_anuales_copia = ventas_anuales.copy()
    for col in ["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS"]:
        ventas_anuales_copia[f"{col}_growth"] = ventas_anuales_copia[col].pct_change() * 100
    
    # Remover el primer año (NaN)
    ventas_anuales_copia = ventas_anuales_copia.dropna()
    
    fig4 = px.bar(
        ventas_anuales_copia,
        x="ANIO",
        y=["VEH_ELECTR_growth", "VEH_HIBRIDAS_PLUGIN_growth", "VEH_HIBRIDAS_growth"],
        title="📈 Crecimiento Anual Porcentual por Tipo",
        labels={"value": "Crecimiento (%)", "variable": "Tipo", "ANIO": "Año"},
        barmode="group"
    )
    fig4.update_layout(height=500)
    fig4.show()
    
    # Mostrar tabla de crecimiento
    print("\n📊 Datos de crecimiento anual (%):")
    crecimiento_display = ventas_anuales_copia[["ANIO", "VEH_ELECTR_growth", "VEH_HIBRIDAS_PLUGIN_growth", "VEH_HIBRIDAS_growth"]].round(1)
    crecimiento_display.columns = ["Año", "Eléctricos (%)", "Plug-in (%)", "Híbridos (%)"]
    display(crecimiento_display)

# ## 🗺️ DISTRIBUCIÓN GEOGRÁFICA

In [ ]:
# 🗺️ Análisis por entidades federativas
if not df.empty and "ID_ENTIDAD" in df.columns:
    print("🗺️ Creando análisis por entidades federativas...")
    
    # Top entidades
    top_entidades = df.groupby("ID_ENTIDAD")[["VEH_ELECTR", "VEH_HIBRIDAS_PLUGIN", "VEH_HIBRIDAS", "TOTAL_ALT"]].sum().reset_index()
    top_entidades = top_entidades.sort_values("TOTAL_ALT", ascending=False).head(15)
    
    fig5 = px.bar(
        top_entidades,
        x="TOTAL_ALT",
        y="ID_ENTIDAD",
        title="🏆 Top 15 Entidades - Total de Vehículos Alternativos",
        orientation='h',
        labels={"TOTAL_ALT": "Total de Vehículos", "ID_ENTIDAD": "Entidad"},
        color="TOTAL_ALT",
        color_continuous_scale="viridis"
    )
    fig5.update_layout(height=600)
    fig5.show()
    
    # Mostrar tabla de top entidades
    print("\n📊 Top 15 entidades:")
    top_entidades_display = top_entidades[["ID_ENTIDAD", "VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN", "TOTAL_ALT"]]
    top_entidades_display.columns = ["Entidad", "Eléctricos", "Híbridos", "Plug-in", "Total"]
    display(top_entidades_display)

In [ ]:
# 🔥 Mapa de calor mejorado por entidad y año
if not df.empty and "ID_ENTIDAD" in df.columns:
    print("🔥 Creando mapa de calor mejorado...")
    
    # Preparar datos
    heatmap_data = df.groupby(["ID_ENTIDAD", "ANIO"])["TOTAL_ALT"].sum().reset_index()
    heatmap_pivot = heatmap_data.pivot(index="ID_ENTIDAD", columns="ANIO", values="TOTAL_ALT").fillna(0)
    
    # Filtrar solo las top 20 entidades
    top_20_entidades = top_entidades.head(20)["ID_ENTIDAD"].tolist()
    heatmap_pivot_filtered = heatmap_pivot.loc[heatmap_pivot.index.isin(top_20_entidades)]
    
    # Crear el mapa de calor con mejor configuración
    fig6 = px.imshow(
        heatmap_pivot_filtered,
        title="🔥 Mapa de Calor: Ventas por Entidad y Año (Top 20)",
        labels=dict(x="Año", y="Entidad", color="Ventas"),
        aspect="auto",
        color_continuous_scale="viridis",
        text_auto=True  # Mostrar valores en las celdas
    )
    
    # Mejorar el layout para mejor legibilidad
    fig6.update_layout(
        height=800,  # Aumentar altura
        width=1000,  # Aumentar ancho
        title={
            'text': "🔥 Mapa de Calor: Ventas por Entidad y Año (Top 20)",
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis={
            'title': 'Año',
            'tickangle': 0,  # Rotar etiquetas del eje X
            'tickmode': 'linear',
            'tick0': heatmap_pivot_filtered.columns.min(),
            'dtick': 1
        },
        yaxis={
            'title': 'ID Entidad',
            'tickmode': 'linear',
            'autorange': 'reversed'  # Para mostrar de mayor a menor
        },
        font=dict(size=12),
        margin=dict(l=100, r=100, t=100, b=100)  # Más margen
    )
    
    # Personalizar el texto en las celdas
    fig6.update_traces(
        texttemplate="%{z:.0f}",  # Formato sin decimales
        textfont={"size": 8},
        hovertemplate="<b>Entidad:</b> %{y}<br>" +
                      "<b>Año:</b> %{x}<br>" +
                      "<b>Ventas:</b> %{z:,.0f}<br>" +
                      "<extra></extra>"
    )
    
    fig6.show()

# 📊 Alternativa: Gráfico de barras agrupadas (más legible)
if not df.empty and "ID_ENTIDAD" in df.columns:
    print("📊 Creando gráfico de barras alternativo...")
    
    # Tomar solo top 10 para mejor visualización
    top_10_entidades = top_entidades.head(10)["ID_ENTIDAD"].tolist()
    df_top10 = df[df["ID_ENTIDAD"].isin(top_10_entidades)]
    
    # Agrupar datos
    barras_data = df_top10.groupby(["ID_ENTIDAD", "ANIO"])["TOTAL_ALT"].sum().reset_index()
    
    # Crear gráfico de barras agrupadas
    fig7 = px.bar(
        barras_data,
        x="ID_ENTIDAD",
        y="TOTAL_ALT",
        color="ANIO",
        title="📊 Ventas por Entidad y Año (Top 10) - Vista Alternativa",
        labels={"TOTAL_ALT": "Ventas", "ID_ENTIDAD": "Entidad"},
        barmode='group',
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    
    fig7.update_layout(
        height=600,
        width=1000,
        title={'x': 0.5, 'xanchor': 'center'},
        xaxis={'tickangle': 45},
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig7.show()

## 📋 ANÁLISIS ESTADÍSTICO Y DATOS

In [ ]:
# 📊 Estadísticas descriptivas
if not df.empty:
    print("📊 Estadísticas descriptivas por tipo de vehículo:")
    stats = df[["VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN", "TOTAL_ALT"]].describe()
    stats.columns = ["Eléctricos", "Híbridos", "Plug-in", "Total Alternativo"]
    display(stats)
    
    # Información adicional del dataset
    print("\n📋 Información del Dataset:")
    info_data = {
        "Métrica": [
            "Total de registros",
            "Período de análisis", 
            "Entidades únicas",
            "Años disponibles",
            "Meses únicos",
            "Valor máximo (Eléctricos)",
            "Valor máximo (Híbridos)",
            "Valor máximo (Plug-in)"
        ],
        "Valor": [
            f"{len(df):,}",
            f"{df['ANIO'].min():.0f} - {df['ANIO'].max():.0f}",
            f"{df['ID_ENTIDAD'].nunique()}" if "ID_ENTIDAD" in df.columns else "N/A",
            f"{len(df['ANIO'].unique())}",
            f"{len(df['ID_MES'].unique())}",
            f"{df['VEH_ELECTR'].max():,}",
            f"{df['VEH_HIBRIDAS'].max():,}",
            f"{df['VEH_HIBRIDAS_PLUGIN'].max():,}"
        ]
    }
    
    info_df = pd.DataFrame(info_data)
    display(info_df)


In [ ]:
# 📈 Gráfica adicional: Evolución mensual agregada
if not df.empty:
    print("📅 Creando evolución mensual agregada...")
    
    ventas_por_mes = df.groupby("ID_MES")[["VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN"]].mean().reset_index()
    
    fig7 = px.bar(
        ventas_por_mes,
        x="ID_MES",
        y=["VEH_ELECTR", "VEH_HIBRIDAS", "VEH_HIBRIDAS_PLUGIN"],
        title="📅 Promedio Mensual de Ventas (Todos los años)",
        labels={"value": "Promedio de Vehículos", "ID_MES": "Mes", "variable": "Tipo"},
        barmode="group"
    )
    fig7.update_layout(height=500)
    fig7.show()

In [ ]:
# 💾 Exportar datos procesados
if not df.empty:
    print("💾 Preparando exportación de datos...")
    
    # Crear resumen ejecutivo
    resumen_ejecutivo = pd.DataFrame({
        'Tipo_Vehiculo': ['Eléctricos', 'Híbridos', 'Plug-in', 'Total'],
        'Total_Ventas': [
            df["VEH_ELECTR"].sum(),
            df["VEH_HIBRIDAS"].sum(), 
            df["VEH_HIBRIDAS_PLUGIN"].sum(),
            df["TOTAL_ALT"].sum()
        ],
        'Promedio_Anual': [
            df.groupby("ANIO")["VEH_ELECTR"].sum().mean(),
            df.groupby("ANIO")["VEH_HIBRIDAS"].sum().mean(),
            df.groupby("ANIO")["VEH_HIBRIDAS_PLUGIN"].sum().mean(),
            df.groupby("ANIO")["TOTAL_ALT"].sum().mean()
        ],
        'Maximo_Anual': [
            df.groupby("ANIO")["VEH_ELECTR"].sum().max(),
            df.groupby("ANIO")["VEH_HIBRIDAS"].sum().max(),
            df.groupby("ANIO")["VEH_HIBRIDAS_PLUGIN"].sum().max(),
            df.groupby("ANIO")["TOTAL_ALT"].sum().max()
        ]
    })
    
    print("📊 Resumen Ejecutivo:")
    display(resumen_ejecutivo)
    
    # Guardar archivos (opcional)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Descomenta las siguientes líneas si quieres guardar los archivos
    # df.to_csv(f"vehiculos_procesados_{timestamp}.csv", index=False)
    # resumen_ejecutivo.to_csv(f"resumen_ejecutivo_{timestamp}.csv", index=False)
    # print(f"✅ Archivos guardados con timestamp: {timestamp}")

## COMPARACION COLOMBIA - MEXICO

In [ ]:

# ==========================================================
# ⚡ Comparación de Vehículos Eléctricos por Año: México vs Colombia
# ==========================================================

import pandas as pd
import plotly.express as px

# -------------------------------
# 🇲🇽 México — combinar datasets de 2016 a 2022
# -------------------------------
anios = list(range(2016, 2023))

dfs_mex = [
    leer_archivo(f"data/conjunto_de_datos/raiavl_hibrido_mensual_tr_cifra_{anio}.csv").assign(ANIO=anio)
    for anio in anios
]

df_mex = pd.concat(dfs_mex, ignore_index=True)

# Asegurar tipo numérico
df_mex['VEH_ELECTR'] = pd.to_numeric(df_mex['VEH_ELECTR'], errors='coerce')
df_mex['ANIO'] = pd.to_numeric(df_mex['ANIO'], errors='coerce')

# Agrupar por año y sumar cantidad de eléctricos
electricos_mex = df_mex.groupby('ANIO')['VEH_ELECTR'].sum().reset_index()
electricos_mex['País'] = 'México'

print("✅ Datos de México listos:", electricos_mex.shape)
display(electricos_mex.head())

# -------------------------------
# 🇨🇴 Colombia — cargar y procesar (2010 a 2022)
# -------------------------------
df_col = leer_archivo("data/num_ev.csv")

# Filtrar solo vehículos eléctricos
df_col = df_col[df_col['COMBUSTIBLE'].str.upper() == 'ELECTRICO']

# Convertir fecha a año
df_col['FECHA_REGISTRO'] = pd.to_datetime(df_col['FECHA_REGISTRO'], errors='coerce')
df_col['ANIO'] = df_col['FECHA_REGISTRO'].dt.year

# Filtrar rango 2010–2022
df_col = df_col[(df_col['ANIO'] >= 2016) & (df_col['ANIO'] <= 2022)]

# Contar vehículos eléctricos por año
electricos_col = df_col.groupby('ANIO').size().reset_index(name='VEH_ELECTR')
electricos_col['País'] = 'Colombia'

print("✅ Datos de Colombia listos:", electricos_col.shape)
display(electricos_col.head())

# -------------------------------
# Combinar ambos países
# -------------------------------
df_comp = pd.concat([electricos_mex, electricos_col], ignore_index=True)

# -------------------------------
# Gráfico comparativo
# -------------------------------
fig = px.line(
    df_comp,
    x='ANIO',
    y='VEH_ELECTR',
    color='País',
    markers=True,
    title='⚡ Comparación de Vehículos Eléctricos por Año — México vs Colombia (2010–2022)',
    labels={'ANIO': 'Año', 'VEH_ELECTR': 'Cantidad de Vehículos Eléctricos'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(
    text=df_comp['VEH_ELECTR'],
    textposition="top center",
    hovertemplate="<b>Año:</b> %{x}<br><b>Vehículos eléctricos:</b> %{y:,}"
)
fig.update_layout(
    height=500,
    font=dict(size=12),
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray', tickformat=',d'),
    plot_bgcolor='white'
)

fig.show()

In [ ]:
# ==========================================================
# ⚡ Comparación de Vehículos Eléctricos por Año: México vs Colombia
# ==========================================================

import pandas as pd
import plotly.express as px

# -------------------------------
# 🇲🇽 México — combinar datasets de 2016 a 2022
# -------------------------------
anios = list(range(2016, 2023))

dfs_mex = [
    leer_archivo(f"data/conjunto_de_datos/raiavl_hibrido_mensual_tr_cifra_{anio}.csv").assign(ANIO=anio)
    for anio in anios
]

df_mex = pd.concat(dfs_mex, ignore_index=True)

# Asegurar tipo numérico
df_mex['VEH_HIBRIDAS'] = pd.to_numeric(df_mex['VEH_HIBRIDAS'], errors='coerce')
df_mex['ANIO'] = pd.to_numeric(df_mex['ANIO'], errors='coerce')

# Agrupar por año y sumar cantidad de eléctricos
electricos_mex = df_mex.groupby('ANIO')['VEH_HIBRIDAS'].sum().reset_index()
electricos_mex['País'] = 'México'

print("✅ Datos de México listos:", electricos_mex.shape)
display(electricos_mex.head())

# -------------------------------
# 🇨🇴 Colombia — cargar y procesar (2010 a 2022)
# -------------------------------
df_col = leer_archivo("data/num_ev.csv")

# Filtrar solo vehículos eléctricos
df_col = df_col[df_col['COMBUSTIBLE'].str.upper() == 'GASO ELEC']

# Convertir fecha a año
df_col['FECHA_REGISTRO'] = pd.to_datetime(df_col['FECHA_REGISTRO'], errors='coerce')
df_col['ANIO'] = df_col['FECHA_REGISTRO'].dt.year

# Filtrar rango 2010–2022
df_col = df_col[(df_col['ANIO'] >= 2016) & (df_col['ANIO'] <= 2022)]

# Contar vehículos eléctricos por año
electricos_col = df_col.groupby('ANIO').size().reset_index(name='VEH_HIBRIDAS')
electricos_col['País'] = 'Colombia'

print("✅ Datos de Colombia listos:", electricos_col.shape)
display(electricos_col.head())

# -------------------------------
# Combinar ambos países
# -------------------------------
df_comp = pd.concat([electricos_mex, electricos_col], ignore_index=True)

# -------------------------------
# Gráfico comparativo
# -------------------------------
fig = px.line(
    df_comp,
    x='ANIO',
    y='VEH_HIBRIDAS',
    color='País',
    markers=True,
    title='⚡ Comparación de Vehículos Hibridos por Año — México vs Colombia (2010–2022)',
    labels={'ANIO': 'Año', 'VEH_HIBRIDAS': 'Cantidad de Vehículos Hibridos'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(
    text=df_comp['VEH_HIBRIDAS'],
    textposition="top center",
    hovertemplate="<b>Año:</b> %{x}<br><b>Vehículos eléctricos:</b> %{y:,}"
)
fig.update_layout(
    height=500,
    font=dict(size=12),
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray', tickformat=',d'),
    plot_bgcolor='white'
)

fig.show()